In [1]:
# src/sh_ssw_methods/u_tend.py
from __future__ import annotations
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from utils.event_iden_funcs import (
    build_events_df
)

In [3]:
from utils.xr_operators import (
    remove_doy_climatology,
)

In [4]:
import os

- detection algorithm

In [29]:
def detect_ssw_u_tend(
    u_daily: xr.DataArray,
    *,
    time_dim: str = "time",
    thres: float = -35.0,          # e.g., -35 (10 hPa), -19 (50 hPa)
    season_month_min: int = 5,     # months condition: (m > 4) & (m < 11) -> 5..10
    season_month_max: int = 10,
    min_gap_days: int = 20,
    persist_days: int = 10,
    rng: int = 7, # +/- 7 days around current day to define u tendency
    level_hpa: float | None = None,
    data_source: str | None = None,
    latitude: str | None = "-60",
) -> tuple[pd.DataFrame, pd.DatetimeIndex]:


    u_anom = remove_doy_climatology(u_daily)

    # To pandas for simple window logic
    t = pd.to_datetime(u_daily[time_dim].to_index())
    s_full = pd.Series(u_daily.values, index=t)
    s_anom = pd.Series(u_anom.values,  index=t)

    mns = t.month
    ndd = len(s_full)

     #Calculate delta
    delta=np.zeros(ndd)
    for i in range(rng,ndd-1-rng):
        delta[i]=s_full.iloc[i+rng]-s_full.iloc[i-rng] 
    
    dates_list = []  # to store datetime objects

    for i in range(rng, ndd - 1 - rng):
        if (delta[i] < thres) & (delta[i-1] >= thres) & (mns[i+1] > 4) & (mns[i+1] < 11):
            min1 = np.amin(delta[i-min_gap_days:i])
            min2 = s_full.iloc[i:i+persist_days].min()
    
            if (min1 > thres) & (min2 > 0):
                date_obj = t[i+1]
                dates_list.append(date_obj)

    event_dates = pd.to_datetime(dates_list)
    
    # output into df
    events_df = build_events_df(
        dates=event_dates,
        method="u_tend", #### change here the name later
        definition=f"u_tend_{int(thres)}m/s_{int(level_hpa)}hPa_{int(np.abs(latitude))}S",
        data_source=data_source or "",
        threshold=f"{int(thres)}m/s",
        level_hpa=str(level_hpa),
        latitude=str(latitude),
        notes=f"u_tend < thres at_least_{persist_days}d; u_tend defined as u_full difference between +/-{rng} days from current day; min_gap={min_gap_days}d",
        # extra_cols={
        # "u_anom": u_anom_event, }, 
    )

    return events_df, event_dates

- test

- for 50hPa

In [30]:
thres = -19
level_hpa = 50
latitude = -60

In [31]:
u5060_path =  os.path.abspath(os.path.join(os.getcwd(), "..", ".."))+ "/data/" +"u5060s_era5_1959_2023.nc"
xu5060 = xr.open_dataset(u5060_path)

In [32]:
da = xu5060['u5060S'].sel(time=slice("1979","2020"))

In [33]:
base_start = "1979-01-01"
base_end = "2020-12-31"
time_dim = "time"

In [34]:
u_daily = da.resample({time_dim: "1D"}).mean()

In [35]:
events_df, event_dates = detect_ssw_u_tend(u_daily, thres=thres, level_hpa=level_hpa, latitude=latitude)

In [36]:
event_dates

DatetimeIndex(['1982-10-01', '1983-10-31', '1984-10-29', '1985-10-29',
               '1988-10-21', '1992-09-26', '1992-10-22', '1993-10-17',
               '1995-10-07', '1997-10-29', '1999-10-30', '2002-09-17',
               '2002-10-29', '2003-10-25', '2004-10-08', '2005-10-05',
               '2009-10-08', '2011-10-30', '2014-10-04', '2014-10-27',
               '2016-09-27', '2017-09-08', '2017-10-29'],
              dtype='datetime64[ns]', freq=None)

- for 10hPa

In [38]:
thres = -35
level_hpa = 10
latitude = -60

In [39]:
u1060_path =  os.path.abspath(os.path.join(os.getcwd(), "..", ".."))+ "/data/" +"u1060s_era5_1959_2023.nc"
xu1060 = xr.open_dataset(u1060_path)

In [40]:
da = xu1060['u1060S'].sel(time=slice("1979","2020"))

In [41]:
base_start = "1979-01-01"
base_end = "2020-12-31"
time_dim = "time"

In [42]:
u_daily = da.resample({time_dim: "1D"}).mean()

In [43]:
events_df, event_dates = detect_ssw_u_tend(u_daily, thres=thres, level_hpa=level_hpa, latitude=latitude)

In [44]:
event_dates

DatetimeIndex(['1980-10-03', '1982-10-02', '1983-10-26', '1984-09-29',
               '1990-09-22', '1992-09-25', '1993-10-18', '1997-10-29',
               '2000-09-30', '2002-08-17', '2003-09-29', '2004-09-12',
               '2005-10-07', '2006-10-15', '2007-09-13', '2008-10-07',
               '2009-09-27', '2012-10-04', '2013-10-15', '2014-10-09',
               '2017-09-07', '2019-09-01'],
              dtype='datetime64[ns]', freq=None)